# Tutorial 06 — Multilingual and low-resource language modelling

[Lecture notes](../lectures/06_multilingual.md) · [Exercise sheet](../exercises/06_multilingual.md)

**40 minutes, CPU, no accounts or model downloads.** Requires Python 3.10+ and NumPy. These are mechanism experiments using authored or synthetic data, not frontier benchmark results. Restart the kernel and run all cells in order.

**Objective:** Expose aggregate metric failures, bootstrap a paired comparison, and inspect language sampling weights.

Allocate 5 minutes to setup, 15 to the mechanism, 10 to interventions, and 10 to discussion. Record predictions before running. Complete the separate exercise sheet after class.


In [1]:
import sys
import numpy as np
rng = np.random.default_rng(42)
print("Python", sys.version.split()[0], "NumPy", np.__version__, "seed", 42)


Python 3.13.1 NumPy 2.1.3 seed 42


## 1. Expose an aggregate metric trap

Language names here label an invented outcome table. These numbers are deliberately synthetic and say nothing about actual model performance.

Before running: predict the output or the invariant being checked.


In [2]:
language=np.array(["English"]*100+["Hausa"]*10)
correct_a=np.array([1]*90+[0]*10+[1]*3+[0]*7)
correct_b=np.array([1]*86+[0]*14+[1]*7+[0]*3)
def report(c):
    slices={l:{"n":int((language==l).sum()),"accuracy":float(c[language==l].mean())} for l in np.unique(language)}
    return {"overall":float(c.mean()),"equal_language":float(np.mean([x["accuracy"] for x in slices.values()])),"slices":slices}
print("A",report(correct_a)); print("B",report(correct_b))
assert np.isclose(correct_a.mean(),correct_b.mean())


A {'overall': 0.8454545454545455, 'equal_language': 0.6, 'slices': {np.str_('English'): {'n': 100, 'accuracy': 0.9}, np.str_('Hausa'): {'n': 10, 'accuracy': 0.3}}}
B {'overall': 0.8454545454545455, 'equal_language': 0.78, 'slices': {np.str_('English'): {'n': 100, 'accuracy': 0.86}, np.str_('Hausa'): {'n': 10, 'accuracy': 0.7}}}


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## 2. Paired bootstrap and metric conventions

We resample the same item indices for A and B. Real speaker/document dependence would require a clustered bootstrap.

Before running: predict the output or the invariant being checked.


In [3]:
idx=rng.integers(0,len(language),size=(2000,len(language)))
diff=(correct_b[idx]-correct_a[idx]).mean(axis=1)
print("B-A observed",float(np.mean(correct_b-correct_a)),"95% percentile interval",np.quantile(diff,[.025,.975]))
def macro_f1(y,p,classes):
    scores=[]
    for c in classes:
        tp=np.sum((y==c)&(p==c)); fp=np.sum((y!=c)&(p==c)); fn=np.sum((y==c)&(p!=c))
        scores.append(2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.)
    return float(np.mean(scores))
y=np.array([0,0,0,1,1,2]); p=np.array([0,0,0,0,1,0])
print("accuracy",np.mean(y==p),"macro F1 (absent-class zero)",macro_f1(y,p,[0,1,2]))


B-A observed 0.0 95% percentile interval [-0.04545455  0.05454545]
accuracy 0.6666666666666666 macro F1 (absent-class zero) 0.47222222222222215


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## 3. Language sampling weights

Decreasing alpha balances exposure, but it cannot create new unique documents.

Before running: predict the output or the invariant being checked.


In [4]:
sizes=np.array([10000,1000,100])
for alpha in [1.,.5,0.]:
    q=sizes**alpha; q=q/q.sum()
    print("alpha",alpha,"sampling",q,"expected repeats in 10000 draws",10000*q/sizes)
    assert np.isclose(q.sum(),1)


alpha 1.0 sampling [0.9009009  0.09009009 0.00900901] expected repeats in 10000 draws [0.9009009 0.9009009 0.9009009]
alpha 0.5 sampling [0.70610111 0.22328878 0.07061011] expected repeats in 10000 draws [0.70610111 2.23288777 7.06101112]
alpha 0.0 sampling [0.33333333 0.33333333 0.33333333] expected repeats in 10000 draws [ 0.33333333  3.33333333 33.33333333]


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## Practical discussion

1. Why might uniform language sampling hurt a very small language?
2. Can translated evaluation data establish performance on naturally occurring code-switching?
3. Why is macro-F1 useful but insufficient for a multilingual system?


## Extension and submission

Run `extensions/afrisenti_baseline.py` on permitted train/dev/test TSV exports. It fits a character TF-IDF classifier without downloading or redistributing tweets.

See the [extension guide](../extensions/README.md) before running model downloads. Extensions require additional dependencies and are not necessary for full exercise credit.

Submit the completed notebook with a 150–250-word interpretation and the separate exercise answers. Include a baseline, one controlled change, data provenance, seed, and an honest limitation. Never replace measured results with expected trends.
